# 🚀 ViSceT5 — PreSTU SplitOCR Pre-Training & Visual Inspection trên Kaggle

Notebook chuẩn hướng dẫn quy trình tiền huấn luyện (**PreSTU SplitOCR**) cho mô hình **ViSceT5** trên 2 bộ dữ liệu Scene-Text tiếng Việt: **VinText** và **EVJVQA**, kèm công cụ trực quan hóa kết quả sinh văn bản, toạ độ Bounding Box và Attention Heatmap.

---

### 📌 Cơ chế phương pháp PreSTU & Cải tiến Spatial Region Clustering:
1. **Chỉ nhận Image Pixels + Text Prompt:** Mô hình encoder nhận ảnh điểm ảnh từ CLIP-ViT và chuỗi văn bản tiền tố (prefix prompt).
2. **Khoanh vùng Cụm Không Gian (Spatial Region Clustering for SplitOCR):**
   * Thay vì cắt chuỗi 1 chiều ngẫu nhiên khiến các từ mục tiêu bị phân tán rải rác khắp 4 góc ảnh, thuật toán chọn một anchor box và gom cụm các hộp lân cận theo khoảng cách hình học có trọng số trục $y$ để tạo thành **Target Region** (khối biển hiệu/dòng chữ cục bộ).
   * Toàn bộ từ trong vùng khoanh là **Target** (Suffix + BBoxes), các từ ngoài vùng là **Input** (Prefix Context). Điều này giúp mô hình chỉ cần tập trung chú ý vào đúng một vùng cục bộ để đọc chữ và dự đoán toạ độ, khắc phục triệt để hiện tượng underfitting do ảnh nhỏ $224 \times 224$.
   * **Phân tách từ tự nhiên:** Các từ được nối bằng khoảng trắng `" "` tự nhiên (theo Figure 4 PreSTU), chỉ kết thúc bằng một token `</s>` duy nhất ở cuối chuỗi mục tiêu.
3. **Dual-Target Optimization:** Cân bằng giữa bài toán Sinh văn bản (Text Generation CE Loss) và Định vị toạ độ không gian (BBox CE Loss qua $1000$ bins toạ độ + Soft-argmax L1 distance):
   $$\mathcal{L} = \mathcal{L}_{\text{text}} + \lambda_{\text{bbox}} \cdot \mathcal{L}_{\text{bbox}} \quad (\lambda_{\text{bbox}} = 0.3)$$
4. **Visual Adaptation Cho Ký Tự Tiếng Việt:** Mở băng **4 lớp cuối** của CLIP ViT (`vision_unfreeze_last_n: 4`) với Differential LR ($1\times 10^{-5}$ vs $1\times 10^{-4}$ của T5) giúp visual encoder thích ứng sâu với dấu thanh và nét chữ tiếng Việt mà không làm hỏng các bộ lọc thị giác cơ bản.
5. **Visual Inspection Tool:** Tích hợp trực quan hóa 3 khung hình: Ground Truth (Prefix + Suffix BBoxes) vs Model Prediction (Text + BBoxes) vs Visual Focus Attention Heatmap.

## 1. Clone Codebase & Checkout Nhánh Pretrain
Đồng bộ repository từ GitHub và chuyển sang nhánh `exp/pretrain-gen-all` chứa các cải tiến mới nhất.

In [ ]:
import os
if not os.path.exists("/kaggle/working/ViSceT5"):
    !git clone https://github.com/Kussssssss/ViSceT5.git /kaggle/working/ViSceT5
%cd /kaggle/working/ViSceT5
!git fetch origin
!git checkout exp/pretrain-gen-all
!git pull origin exp/pretrain-gen-all
!git log --oneline -3

## 2. Cài Đặt Môi Trường Chuẩn (Transformers 4.45.2)
Gỡ các phiên bản thư viện mặc định của Kaggle và cài đặt chính xác các phiên bản tương thích từ `requirements.txt`.

In [ ]:
%%capture
!pip uninstall -y transformers peft accelerate 2>/dev/null || true
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/salaniz/pycocoevalcap

In [ ]:
# Kiểm tra xác nhận phiên bản môi trường
import torch
import transformers
print(f"✅ PyTorch Version: {torch.__version__} (CUDA Available: {torch.cuda.is_available()})")
print(f"✅ Transformers Version: {transformers.__version__} (Yêu cầu cố định: 4.45.2)")
if torch.cuda.is_available():
    print(f"✅ GPU Device: {torch.cuda.get_device_name(0)} (Count: {torch.cuda.device_count()})")
assert transformers.__version__.startswith("4.45"), f"Cảnh báo: Cần transformers 4.45.x để khớp kiến trúc module, hiện tại là {transformers.__version__}"

## 3. Chuẩn Bị Dữ Liệu (VinText + EVJVQA)
Tự động tải các file nén Image và OCR từ Google Drive theo cấu hình `configs/data/VinText.yaml` và `configs/data/EVJVQA.yaml`, ghép cặp ảnh-OCR và lưu cache.

In [ ]:
!python scripts/prepare_dataset.py --config configs/data/VinText.yaml,configs/data/EVJVQA.yaml

## 4. Khởi Tạo Trọng Số Mô Hình (ViT5 Base & CLIP ViT)
Khởi tạo `OpenViVQAModel`, tải các trọng số nền tảng ViT5 và CLIP-ViT, kiểm tra tính toàn vẹn số học.

In [ ]:
!python scripts/init_model.py

## 5. Chạy Huấn Luyện PreSTU SplitOCR Pre-Training

### Cấu hình tối ưu theo phương pháp PreSTU:
* **Epochs:** 10
* **Batch size:** 4 (per device) $\times$ 4 (gradient accumulation) = Effective Batch Size 16
* **Learning rate:** $1\times 10^{-4}$ (ViT5) và $1\times 10^{-5}$ (CLIP ViT unfrozen 4 layers)
* **Loss balance:** $\lambda_{\text{bbox}} = 0.3$
* **Output dir:** `/kaggle/working/pretrain_output`

In [ ]:
!python training/pretrain.py configs/pretrain.yaml \
    --dataset_name "VinText,EVJVQA" \
    --num_train_epochs 10 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 0.0001 \
    --lambda_bbox_ce 0.3 \
    --vision_unfreeze_last_n 4 \
    --save_total_limit 1 \
    --output_dir /kaggle/working/pretrain_output \
    --logging_dir /kaggle/working/pretrain_output/logs

## 6. Trực Quan Hóa Kết Quả & Attention Heatmap (Interactive Visual Inspection)

Cell này tải checkpoint vừa huấn luyện và trực quan hóa 3 khung hình song song:
1. **Khung 1 (Ground-Truth):** Ảnh gốc + Hộp Bounding Box Prefix (Xanh dương) + Hộp Bounding Box Suffix Mục tiêu (Xanh lá) + Chuỗi từ Suffix chuẩn.
2. **Khung 2 (Model Prediction):** Ảnh gốc + Hộp Bounding Box Suffix mô hình dự đoán (Đỏ) + Chuỗi từ Suffix do ViT5 Decoder sinh ra qua Beam Search.
3. **Khung 3 (Visual Focus Attention Heatmap):** Bản đồ nhiệt chú ý không gian của Visual Search (AVF) đè lên ảnh gốc, chỉ rõ vùng mắt mô hình đang tập trung nhìn khi sinh từ vựng.

In [ ]:
import sys
import os

# Import hàm trực quan hóa PreSTU SplitOCR
from scripts.visualize_pretrain import visualize_pretrain_samples

figs = visualize_pretrain_samples(
    checkpoint="/kaggle/working/pretrain_output",
    val_csv=None,  # Tự động định vị merged_val.csv
    sample_idx=0,
    num_samples=5,
    save_dir="/kaggle/working/pretrain_output/visualizations",
    show_plot=True
)

print(f"\n✅ Đã tạo và hiển thị thành công {len(figs)} mẫu trực quan hóa!")

## 7. Nén Checkpoint Để Tải Về & Tùy Chọn Upload HuggingFace

In [ ]:
# Nén toàn bộ checkpoint pretrain và ảnh visualizations thành file ZIP để tải về từ giao diện Kaggle
!zip -r /kaggle/working/ViSceT5_PreSTU_Pretrain.zip /kaggle/working/pretrain_output
print("✅ Đã nén thành công checkpoint tại /kaggle/working/ViSceT5_PreSTU_Pretrain.zip")

In [ ]:
# (Tùy chọn) Đăng tải trực tiếp checkpoint lên HuggingFace Hub nếu có Token
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path="/kaggle/working/pretrain_output",
#     repo_id="your-username/ViSceT5-PreSTU-Pretrained",
#     repo_type="model",
#     token="your_hf_token_here"
# )